# Notebook 3 — Validation Bridge: Strategic Annual Severity × Daily Pixel Alerts

## Goal

This notebook explains the **validation bridge** used in the Streamlit dashboard.

The bridge connects the two models without changing their purposes:

1. **Strategic annual model**  
   Governorate-year severity: `Low`, `Medium`, `High`

2. **Operational pixel model**  
   Daily 500 m alert ranking: `Critical`, `High`, `Watch`, `Monitor`

The bridge aggregates daily pixel-alert pressure into governorate-year summaries, then compares those summaries with annual severity predictions.

## Why this bridge matters

It answers:

> Do governorates that are strategically severe also show more daily pixel alert pressure?

It is not a third predictive model. It is a validation and interpretation layer.


## 1. Imports and paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd()

def find_artifact(names, folders=("data", "models", ".")) -> Path:
    """Find a project file in either a flat repo layout or data/models folders."""
    if isinstance(names, str):
        names = [names]
    search_dirs = []
    for folder in folders:
        p = ROOT / folder if folder != "." else ROOT
        if p not in search_dirs:
            search_dirs.append(p)
    for folder in search_dirs:
        for name in names:
            candidate = folder / name
            if candidate.exists():
                return candidate
    # Return the first preferred name in the root so error messages are clear.
    return ROOT / names[0]

def require_file(path: Path, purpose: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {purpose}: {path}")

def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    return json.loads(path.read_text(encoding="utf-8"))

def display_if_available(obj, max_rows=10):
    """Small helper for notebooks that may run in different environments."""
    try:
        display(obj.head(max_rows) if hasattr(obj, "head") else obj)
    except NameError:
        print(obj.head(max_rows) if hasattr(obj, "head") else obj)


In [ ]:
import plotly.express as px

ANNUAL_PREDICTIONS_PATH = find_artifact("annual_governorate_predictions.csv")
PIXEL_SCORES_PATH = find_artifact("operational_pixel_scores_all.csv")
BRIDGE_PATH = find_artifact("bridge_annual_pixel_alerts.csv")

BRIDGE_COUNT_COLUMNS = [
    "scored_pixel_days",
    "observed_fire_pixel_days",
    "critical_alert_pixel_days",
    "high_or_critical_alert_pixel_days",
    "watch_or_above_alert_pixel_days",
]
BRIDGE_SCORE_COLUMNS = ["mean_raw_risk_score", "max_raw_risk_score"]
RISK_CLASS_ORDER = ["Low", "Medium", "High"]

print("Annual predictions:", ANNUAL_PREDICTIONS_PATH)
print("Pixel scores:", PIXEL_SCORES_PATH)
print("Bridge output:", BRIDGE_PATH)


## 2. Load the two model outputs

The bridge needs:

- annual governorate predictions
- scored pixel-date outputs

The pixel table must include at least:

- `gouvernorat`
- `year` or `prediction_date`
- `pixel_id_500m`
- `target_fire_1d`
- `raw_risk_score`
- `alert_tier`


In [ ]:
require_file(ANNUAL_PREDICTIONS_PATH, "annual predictions")
require_file(PIXEL_SCORES_PATH, "pixel scores")

annual = pd.read_csv(ANNUAL_PREDICTIONS_PATH)
pixel = pd.read_csv(PIXEL_SCORES_PATH, parse_dates=["prediction_date"])

if "year" not in pixel.columns:
    pixel["year"] = pixel["prediction_date"].dt.year

annual["year"] = pd.to_numeric(annual["year"], errors="coerce").astype("Int64")
pixel["year"] = pd.to_numeric(pixel["year"], errors="coerce").astype("Int64")
annual["gouvernorat"] = annual["gouvernorat"].astype(str).str.strip()
pixel["gouvernorat"] = pixel["gouvernorat"].astype(str).str.strip()

print(f"Annual rows: {len(annual):,}")
print(f"Pixel score rows: {len(pixel):,}")
display_if_available(annual)
display_if_available(pixel)


## 3. Aggregate daily pixel alerts to governorate-year level

The bridge creates annual alert-pressure indicators:

- total scored pixel-days
- observed fire pixel-days
- mean and maximum raw risk score
- Critical pixel-days
- High + Critical pixel-days
- Watch or above pixel-days


In [ ]:
def build_bridge_dataset(annual: pd.DataFrame, pixel: pd.DataFrame) -> pd.DataFrame:
    pixel_for_agg = pixel.dropna(subset=["gouvernorat", "year"]).copy()

    agg = (
        pixel_for_agg.groupby(["gouvernorat", "year"], dropna=False)
        .agg(
            scored_pixel_days=("pixel_id_500m", "count"),
            observed_fire_pixel_days=("target_fire_1d", "sum"),
            mean_raw_risk_score=("raw_risk_score", "mean"),
            max_raw_risk_score=("raw_risk_score", "max"),
            critical_alert_pixel_days=("alert_tier", lambda s: int((s == "Critical").sum())),
            high_or_critical_alert_pixel_days=("alert_tier", lambda s: int(s.isin(["Critical", "High"]).sum())),
            watch_or_above_alert_pixel_days=("alert_tier", lambda s: int(s.isin(["Critical", "High", "Watch"]).sum())),
        )
        .reset_index()
    )

    annual_columns = [
        "gouvernorat", "year", "risk_class", "predicted_risk_class",
        "risk_score", "predicted_severity_score",
        "predicted_fire_count", "predicted_area_burned_ha",
    ]
    annual_columns = [c for c in annual_columns if c in annual.columns]

    bridge = annual[annual_columns].merge(agg, on=["gouvernorat", "year"], how="left")
    bridge["pixel_coverage_available"] = bridge["scored_pixel_days"].notna()
    bridge["pixel_coverage_status"] = np.where(
        bridge["pixel_coverage_available"], "Available", "Not available"
    )
    return bridge

bridge = build_bridge_dataset(annual, pixel)
display_if_available(bridge, 20)


## 4. Coverage handling

A previous dashboard issue came from treating missing pixel coverage as numeric values in charts.

The corrected bridge logic:

- keeps uncovered governorates in the table
- labels them as `Not available`
- excludes them from the scatter chart
- does not pass `NaN` values as Plotly bubble sizes


In [ ]:
def clean_bridge_for_display(bridge: pd.DataFrame) -> pd.DataFrame:
    out = bridge.copy()
    out["year"] = pd.to_numeric(out["year"], errors="coerce").astype("Int64")
    out["pixel_coverage_available"] = out["scored_pixel_days"].notna()
    out["pixel_coverage_status"] = np.where(
        out["pixel_coverage_available"], "Available", "Not available"
    )
    for column in BRIDGE_COUNT_COLUMNS + BRIDGE_SCORE_COLUMNS:
        if column in out.columns:
            out[column] = pd.to_numeric(out[column], errors="coerce")
    return out

bridge = clean_bridge_for_display(bridge)

coverage_summary = (
    bridge.groupby(["year", "pixel_coverage_status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
display_if_available(coverage_summary, 20)


## 5. Bridge chart

The chart compares:

- x-axis: annual severity score
- y-axis: High + Critical daily alert pressure
- bubble size: observed fire pixel-days
- color: predicted annual risk class

Rows without pixel coverage are excluded from the chart but kept in the table.


In [ ]:
def plot_bridge_for_year(bridge: pd.DataFrame, selected_year: int):
    view = bridge[bridge["year"].astype("Int64") == int(selected_year)].copy()

    for column in BRIDGE_COUNT_COLUMNS + BRIDGE_SCORE_COLUMNS:
        if column not in view.columns:
            view[column] = np.nan
        view[column] = pd.to_numeric(view[column], errors="coerce")

    covered_view = view[view["pixel_coverage_available"]].copy()
    if covered_view.empty:
        print(f"No pixel coverage for year {selected_year}.")
        return None, view

    x_column = "predicted_severity_score" if "predicted_severity_score" in covered_view.columns else "risk_score"
    covered_view[x_column] = pd.to_numeric(covered_view[x_column], errors="coerce")
    covered_view["plot_marker_size"] = covered_view["observed_fire_pixel_days"].fillna(0).clip(lower=0) + 1
    plot_view = covered_view.dropna(subset=[x_column, "high_or_critical_alert_pixel_days"]).copy()

    if plot_view.empty:
        print("No valid numeric rows for the bridge chart.")
        return None, view

    fig = px.scatter(
        plot_view,
        x=x_column,
        y="high_or_critical_alert_pixel_days",
        size="plot_marker_size",
        color="predicted_risk_class",
        hover_name="gouvernorat",
        hover_data={
            "observed_fire_pixel_days": True,
            "scored_pixel_days": True,
            "mean_raw_risk_score": ":.3f",
            "plot_marker_size": False,
        },
        category_orders={"predicted_risk_class": RISK_CLASS_ORDER},
        title=f"Annual severity versus daily alert pressure — {selected_year}",
        labels={
            x_column: "Predicted annual severity score" if x_column == "predicted_severity_score" else "Annual severity score",
            "high_or_critical_alert_pixel_days": "Daily pixel alert pressure: High + Critical pixel-days",
        },
    )
    fig.update_traces(marker={"opacity": 0.72, "line": {"width": 1, "color": "white"}})
    fig.update_layout(height=500)
    return fig, view

covered_years = sorted(bridge.loc[bridge["pixel_coverage_available"], "year"].dropna().astype(int).unique())
selected_year = covered_years[-1] if covered_years else int(bridge["year"].dropna().max())
fig, bridge_view = plot_bridge_for_year(bridge, selected_year)

if fig is not None:
    fig.show()

display_if_available(bridge_view.sort_values("high_or_critical_alert_pixel_days", ascending=False, na_position="last"), 20)


## 6. Export the bridge table

This table is used by the dashboard page **Annual–pixel bridge**.

It should be regenerated whenever annual predictions or pixel scores change.


In [ ]:
BRIDGE_PATH.parent.mkdir(parents=True, exist_ok=True)
bridge.to_csv(BRIDGE_PATH, index=False)
print(f"Saved bridge table: {BRIDGE_PATH}")

# Minimal validation.
required_columns = {
    "gouvernorat",
    "year",
    "predicted_risk_class",
    "scored_pixel_days",
    "observed_fire_pixel_days",
    "high_or_critical_alert_pixel_days",
    "pixel_coverage_status",
}
missing = sorted(required_columns - set(bridge.columns))
if missing:
    raise ValueError(f"Bridge table is missing required columns: {missing}")
else:
    print("Bridge table schema is valid.")


## 7. Interpretation

The bridge should be interpreted as a **consistency and monitoring layer**.

Strong agreement means that strategic severity and daily alert pressure are pointing in the same direction.

Disagreement can be useful:

- high annual severity but low daily pixel alerts may indicate missing pixel coverage or a model blind spot
- low annual severity but high alert pressure may indicate emerging localized risk
- missing pixel coverage should not be interpreted as zero risk
